# Tome C4, Chapter 3: Noisy Channels (Python companion)

Exercises first, solutions at the end. We measure how badly repetition codes do, compute capacities with the Blahut–Arimoto algorithm, and then do what Shannon only imagined in 1948: draw random linear codes, decode them by brute force, and watch them beat repetition below capacity and collapse above it.

*Before each solution, predict what the output will look like.*

In [ ]:
import math, random
from math import comb, log2
import numpy as np
import matplotlib.pyplot as plt
random.seed(3); rng = np.random.default_rng(3)

def H(p):
    return -sum(x * log2(x) for x in p if x > 0)

def h(t):
    return H([t, 1 - t])

## Exercises

**P3.1 (★) Repetition.** Compute the error probability of majority decoding for the repetition codes of odd length $n$ on $\mathrm{BSC}(0.1)$; find the first $n$ with error below $10^{-9}$ (Exercise 3.1) and its rate.

**P3.2 (★) Capacities.** Plot $1-h(p)$ and $1-\varepsilon$ (Figure 3.1), check $h(p)\ge2p$ on $[0,\frac12]$ (Exercise 3.2), and evaluate the weak converse bound of Exercise 3.7.

**P3.3 (★★) Blahut–Arimoto.** Implement the algorithm of Section 3.6 for any channel matrix. Run it on the Z-channel (Exercise 3.3), on $\mathrm{BSC}(0.1)$ and on a non-symmetric $3\times3$ channel; check the bound $\frac1t\log_2|A|$ on the error.

**P3.4 (★★) Cascades and parallel channels.** Check Exercises 3.5 and 3.6 numerically, using your Blahut–Arimoto code for the parallel channel.

**P3.5 (★★★) Random linear codes.** Draw random generator matrices $G$ (Theorem 3.8), decode by brute-force nearest codeword, and estimate the error probability on $\mathrm{BSC}(0.1)$ at rate $\frac13$ for growing $n$. Compare with the repetition code of rate $\frac13$. Then do the same at rate $0.6>C$ (Theorem 3.10).

**P3.6 (★) Sphere packing.** Compare the Hamming bound of Exercise 3.8, $\frac1n\log_2\bigl(2^n/\sum_{j\le t}\binom nj\bigr)$ with $t=\lfloor\tau n\rfloor$, to its limit $1-h(\tau)$.

## Solutions

### P3.1

In [ ]:
def rep_error(n, p):
    return sum(comb(n, j) * p**j * (1 - p)**(n - j) for j in range(n // 2 + 1, n + 1))

for n in (3, 5, 7):
    print("n=%d: error %.5f" % (n, rep_error(n, 0.1)))
n = 1
while rep_error(n, 0.1) >= 1e-9:
    n += 2
print("first odd n with error < 1e-9: n=%d, error %.2e, rate %.3f" % (n, rep_error(n, 0.1), 1 / n))
print("(n=%d gives %.2e)" % (n - 2, rep_error(n - 2, 0.1)))

A rate of $0.029$, against a capacity of $0.531$. Repetition buys reliability at a ruinous price.

### P3.2

In [ ]:
t = np.linspace(1e-4, 0.5, 500)
ht = -t*np.log2(t) - (1-t)*np.log2(1-t)
print("min of h(p) - 2p on [0, 1/2]: %.2e" % (ht - 2*t).min())
C = 1 - h(0.1)
print("C = 1 - h(0.1) = %.4f" % C)
R = 0.6
print("weak converse at R=0.6: P_e >= 1 - C/R = %.4f; for n=100: %.4f (1/(nR) = %.4f)" % (1 - C/R, 1 - C/R - 1/(100*R), 1/(100*R)))
plt.figure(figsize=(5, 3.2))
plt.plot(t, 1 - ht, label="BSC: 1-h(p)"); plt.plot(t, 1 - t, label="BEC: 1-eps")
plt.plot([0.1], [C], "o"); plt.xlabel("p or eps"); plt.ylabel("capacity"); plt.legend(); plt.show()

### P3.3

In [ ]:
def blahut_arimoto(W, iters=2000, tol=1e-13):
    # W[x, y] = W(y | x); returns (capacity in bits, optimal input law, list of I(p_t, W))
    W = np.asarray(W, float)
    p = np.full(W.shape[0], 1 / W.shape[0])
    hist = []
    for _ in range(iters):
        q = p @ W
        with np.errstate(divide="ignore", invalid="ignore"):
            D = np.where(W > 0, W * np.log2(W / q), 0).sum(axis=1)   # D(W(.|x) || q)
        hist.append(float(p @ D))
        if D.max() - p @ D < tol:      # max_x D - I is an upper bound on C - I
            break
        p = p * np.exp2(D); p /= p.sum()
    return hist[-1], p, hist

Z = [[1, 0], [0.5, 0.5]]
Cz, pz, hz = blahut_arimoto(Z)
print("Z-channel: C = %.6f, log2(5/4) = %.6f, P(X=1) = %.6f" % (Cz, log2(5/4), pz[1]))
Cb, pb, _ = blahut_arimoto([[0.9, 0.1], [0.1, 0.9]])
print("BSC(0.1): C = %.6f, input law %s" % (Cb, np.round(pb, 6)))
W3 = [[0.7, 0.2, 0.1], [0.1, 0.8, 0.1], [0.3, 0.3, 0.4]]
C3, p3, h3 = blahut_arimoto(W3)
print("3x3 channel: C = %.6f, input law %s" % (C3, np.round(p3, 4)))
ok = all(C3 - It <= log2(3) / (t + 1) + 1e-12 for t, It in enumerate(h3))
print("error after t steps <= log2|A|/t throughout:", ok, "| steps used:", len(h3))

The algorithm finds $P(X=1)=\frac25$ and $\log_2\frac54$ for the Z-channel, and the uniform law for the symmetric BSC. On the $3\times3$ example the third input, the noisiest, gets the least weight.

### P3.4

In [ ]:
for p, q in ((0.1, 0.1), (0.1, 0.2), (0.05, 0.3)):
    r = p + q - 2*p*q
    print("BSC(%.2f) then BSC(%.2f) = BSC(%.3f): capacities %.4f, %.4f -> %.4f" % (p, q, r, 1-h(p), 1-h(q), 1-h(r)))
W1 = np.array([[1, 0], [0.5, 0.5]]); W2 = np.array([[0.9, 0.1], [0.1, 0.9]])
Wpar = np.kron(W1, W2)          # input (x1, x2), output (y1, y2)
Cp, pp, _ = blahut_arimoto(Wpar)
print("parallel Z + BSC(0.1): %.6f = %.6f + %.6f = %.6f" % (Cp, Cz, Cb, Cz + Cb))

### P3.5

In [ ]:
def random_code_error(n, k, p, trials):
    # random k x n generator matrix; codewords packed as integers; nearest-codeword decoding
    G = rng.integers(0, 2, size=(k, n))
    msgs = ((np.arange(2**k)[:, None] >> np.arange(k)) & 1)
    cw = (msgs @ G) % 2
    packed = (cw << np.arange(n)).sum(axis=1).astype(np.uint64)
    errors = 0
    for _ in range(trials):
        m = rng.integers(2**k)
        e = (rng.random(n) < p).astype(np.int64)
        y = np.uint64(int(((cw[m] ^ e) << np.arange(n)).sum()))
        d = np.bitwise_count(packed ^ y)
        best = np.flatnonzero(d == d.min())
        if packed[rng.choice(best)] != packed[m]:   # wrong codeword (ties broken at random)
            errors += 1
    return errors / trials

print("rate 1/3 on BSC(0.1); repetition n=3 has error %.4f" % rep_error(3, 0.1))
for n in (12, 24, 36, 48):
    print("  random linear code n=%2d, k=%2d: error ~ %.4f" % (n, n // 3, random_code_error(n, n // 3, 0.1, 2000)))
print("rate 0.6 > C = 0.531 on BSC(0.1):")
for n in (10, 20, 30):
    k = round(0.6 * n)
    print("  random linear code n=%2d, k=%2d: error ~ %.3f" % (n, k, random_code_error(n, k, 0.1, 400)))

Below capacity the error of random codes falls steadily with $n$; at these short lengths they only just catch up with the length-3 repetition code, whose error is stuck at $0.028$ for this rate, but the theorem says the fall continues to $0$. Above capacity the error climbs towards $1$, as the strong converse demands. Why stop at $n=48$? Brute-force decoding costs $2^k$ comparisons per received word: the riddle of the chapter's introduction, in one line of code.

### P3.6

In [ ]:
for tau in (0.05, 0.1, 0.2):
    vals = []
    for n in (50, 200, 1000, 5000):
        t = int(tau * n)
        vol = sum(comb(n, j) for j in range(t + 1))
        vals.append(1 - log2(vol) / n)
    print("tau=%.2f: Hamming bound %s -> limit 1-h(tau) = %.4f" % (tau, " ".join("%.4f" % v for v in vals), 1 - h(tau)))

The bound decreases to $1-h(\tau)$ from above, with an error of order $\frac{\log n}n$.